# Training runs on Kaggle Notebooks (free GPU)

Trains the VGG models for Figure 1 (with / without BatchNorm) and
Figure 2 (noisy BatchNorm) of Santurkar et al. (2018).

**Settings (right panel → Session options):**
* Accelerator: **GPU T4 x2** (only one GPU is used).
* Internet: **On** (needed to clone the repository and download CIFAR-10;
  Kaggle asks for phone verification the first time).

**How to run:** click **Save Version → Save & Run All (Commit)**.
The notebook then runs in the background (up to 12 hours) and you can
close the browser. Results and checkpoints are written to
`/kaggle/working/runs` and kept as the version's **Output**.

**Resuming:** if a version stops before all runs finish, open the
notebook, click **Add Input**, add the stopped version's output, and
run again. Section 3 copies the old results back and unfinished runs
continue from their last checkpoint (`--resume`).

One 15,000-step run takes roughly 40 minutes on a T4.

## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Get the code
The repository is cloned to `/tmp`, so it does not end up in the notebook output. CIFAR-10 is downloaded to `/tmp/data`.

In [ ]:
import os

REPO_URL = "https://github.com/Sila402/batch-normalization-reproduction.git"
REPO_DIR = "/tmp/batch-normalization-reproduction"

if not os.path.exists(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q

%cd {REPO_DIR}
!git log --oneline -1

## 3. Output folder (and earlier results, if attached as input)

In [ ]:
import shutil
from pathlib import Path

OUT = Path("/kaggle/working/runs")
OUT.mkdir(parents=True, exist_ok=True)

# Copy run folders from an attached earlier output, if any.
for previous in Path("/kaggle/input").glob("**/runs/*_lr*_seed*"):
    target = OUT / previous.name
    if not target.exists():
        shutil.copytree(previous, target)
        print("Restored:", previous.name)

print("Results folder:", OUT)

## 4. Choose the runs

Each entry is `(model, learning rate, seed)`; `model` is `with_bn`,
`without_bn` or `noisy_bn`. Each group member uses their own seed.
Everyone must use the **same** `STEPS`.

In [ ]:
STEPS = 15000

RUNS = [
    ("with_bn",    0.1, 0),
    ("without_bn", 0.1, 0),
    ("noisy_bn",   0.1, 0),
]

## 5. Train
Finished runs are skipped; unfinished runs resume from `latest.pt`.

In [ ]:
import csv
import subprocess
import sys


def run_dir(model, lr, seed):
    return OUT / f"{model}_lr{lr}_seed{seed}"


def finished_steps(model, lr, seed):
    csv_path = run_dir(model, lr, seed) / "metrics.csv"
    if not csv_path.exists():
        return 0
    with open(csv_path, newline="") as f:
        rows = list(csv.DictReader(f))
    return int(rows[-1]["step"]) if rows else 0


for model, lr, seed in RUNS:
    if finished_steps(model, lr, seed) >= STEPS:
        print(f"SKIP {model} lr={lr} seed={seed}: already finished")
        continue

    latest = run_dir(model, lr, seed) / "checkpoints" / "latest.pt"
    command = [
        sys.executable, "-m", "final_reproduction.experiments.train_figure1",
        "--model", model, "--lr", str(lr), "--seed", str(seed),
        "--steps", str(STEPS), "--out", str(OUT),
    ]
    if latest.exists():
        command.append("--resume")

    print("\nRUN:", " ".join(command[2:]), flush=True)
    subprocess.run(command, check=True)

## 6. Summary of finished runs
Final test accuracy and the mean of the last 10 evaluations (last 1,000 steps).

In [ ]:
import pandas as pd

summary = []
for model, lr, seed in RUNS:
    csv_path = run_dir(model, lr, seed) / "metrics.csv"
    if not csv_path.exists():
        continue
    df = pd.read_csv(csv_path)
    summary.append({
        "model": model, "lr": lr, "seed": seed,
        "last_step": int(df["step"].iloc[-1]),
        "final_test_acc": df["test_accuracy"].iloc[-1],
        "mean_last10_test_acc": df["test_accuracy"].tail(10).mean(),
        "final_train_loss": df["train_loss"].iloc[-1],
    })

summary = pd.DataFrame(summary)
summary.to_csv(OUT / "summary.csv", index=False)
summary